In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

def objective(trial):
    # 1. Định nghĩa không gian tìm kiếm (Search Space)
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'enable_categorical': True,
        'random_state': 42,
        'tree_method': 'hist',
        'n_jobs': -1
    }

    # 2. Khởi tạo Cross-Validation theo thời gian
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_train = weights[train_idx]
        
        model = XGBRegressor(**param)
        model.fit(X_train, y_train, sample_weight=w_train)
        
        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        scores.append(mae)

    # Trả về trung bình MAE để Optuna tối thiểu hóa (minimize)
    return np.mean(scores)

# 3. Chạy tối ưu hóa
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50) # Thử 50 tổ hợp khác nhau

print("--- KẾT QUẢ TỐI ƯU HÓA ---")
print(f"MAE tốt nhất: {study.best_value:,.2f}")
print(f"Bộ tham số tốt nhất: {study.best_params}")

In [ ]:
df_all = df_all.sort_values('Date').reset_index(drop=True)

# Độ trễ (Lag) của traffic: Lượng truy cập ngày hôm qua, 7 ngày trước
df_all['lag_1_sessions'] = df_all['total_sessions'].shift(1)
df_all['lag_7_sessions'] = df_all['total_sessions'].shift(7)

# Trung bình trượt (Rolling Window): Trung bình truy cập 7 ngày qua
df_all['rolling_7_sessions'] = df_all['total_sessions'].rolling(window=7).mean()

In [ ]:
best_params_rev = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05, 
    'num_leaves': 86,      
    'max_depth': 10,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1
}

best_params_cogs = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03, 
    'num_leaves': 40,
    'random_state': 42,
    'verbose': -1
}


In [ ]:
final_df['lag_1_sessions'] = final_df['sessions'].shift(1)
final_df['lag_7_sessions'] = final_df['sessions'].shift(7)
final_df['lag_1_items_sold'] = final_df['total_items_sold'].shift(1)
final_df['lag_7_items_sold'] = final_df['total_items_sold'].shift(7)
final_df['lag_364_items_sold'] = final_df['total_items_sold'].shift(364)
final_df['rolling_30_items_mean'] = final_df['total_items_sold'].rolling(window=30).mean()

# 3. Đặc trưng Trượt (Rolling Features)
# Trung bình lưu lượng truy cập trong 7 ngày gần nhất
final_df['rolling_7_sessions_mean'] = final_df['sessions'].rolling(window=7).mean()
# Độ lệch chuẩn của doanh số trong 7 ngày (đo lường biến động)
final_df['rolling_7_items_std'] = final_df['total_items_sold'].rolling(window=7).std()

In [ ]:
# Doanh thu cùng kỳ 2 năm trước
df_calendar['rev_lag_2_years'] = df_calendar['Revenue'].shift(728)

# Xu hướng 1 tháng của cùng kỳ 2 năm trước (Rất mạnh để chống nhiễu)
df_calendar['rev_rolling_30_lag_2_years'] = df_calendar['Revenue'].shift(728).rolling(window=30).mean()

In [ ]:
df_calendar['cogs_lag_2_years'] = df_calendar['COGS'].shift(728)

# Xu hướng 1 tháng của cùng kỳ 2 năm trước (Rất mạnh để chống nhiễu)
df_calendar['cogs_rolling_30_lag_2_years'] = df_calendar['COGS'].shift(728).rolling(window=30).mean()

In [ ]:
daily_features = item_with_date.groupby('order_date').agg(
    total_items_sold=('quantity', 'sum'),           # Tổng sản phẩm bán ra
    total_discount=('discount_value', 'sum'),       # Tổng giá trị đã giảm
    promoted_items_count=('is_promoted', 'sum'),    # Số lượng sản phẩm có áp mã
    unique_orders=('order_id', 'nunique')           # Tổng số lượng đơn hàng
).reset_index()

daily_features = daily_features.rename(columns={'order_date': 'Date'})

In [ ]:
promos['start_date'] = pd.to_datetime(promos['start_date'])
promos['end_date'] = pd.to_datetime(promos['end_date'])

# Quét qua từng ngày trong df_calendar để tìm "Sức ép khuyến mãi đã lên lịch"
planned_promo_stats = []
for current_date in df_calendar['Date']:
    # Tìm các khuyến mãi đang active trong ngày này
    active = promos[(promos['start_date'] <= current_date) & (promos['end_date'] >= current_date)]
    if len(active) > 0:
        planned_discount_depth = active['discount_value'].max() # Mức giảm giá sâu nhất
        active_campaigns = len(active) # Số lượng chiến dịch song song
    else:
        planned_discount_depth = 0
        active_campaigns = 0
        
    planned_promo_stats.append({
        'Date': current_date,
        'planned_discount_depth': planned_discount_depth,
        'active_campaigns': active_campaigns
    })

# Merge planned_promo_stats vào df_calendar